# 01 — Portfolio Composition & Concentration

**Investment question:** How is the portfolio constructed, and how concentrated is it?

This notebook reads the curated `analytics_positions` table created by KNIME. Symbols held across multiple IBKR models are consolidated before portfolio-level concentration is calculated. Monetary outputs and credentials are never committed.

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from sqlalchemy import create_engine

sns.set_theme(style="whitegrid")
load_dotenv()
engine = create_engine(os.environ["DATABASE_URL"])

## Load and validate curated positions

In [ ]:
query = """
SELECT model, symbol, asset_class, currency, quantity, market_value, portfolio_weight
FROM analytics_positions
"""
positions = pd.read_sql(query, engine)
assert positions["symbol"].notna().all()
assert positions[["market_value", "portfolio_weight"]].notna().all().all()
assert abs(positions["portfolio_weight"].sum() - 1) < 1e-4
print(f"Rows: {len(positions):,}")
print(f"Unique symbols: {positions['symbol'].nunique():,}")

## Allocation by IBKR model

In [ ]:
model_allocation = (
    positions.groupby("model", dropna=False)
    .agg(position_rows=("symbol", "size"), unique_symbols=("symbol", "nunique"), weight=("portfolio_weight", "sum"))
    .sort_values("weight", ascending=False)
)
model_allocation.assign(weight_pct=lambda x: x["weight"] * 100)

## Consolidate symbols held across models
A symbol may legitimately appear in more than one model. Its weights are summed for portfolio-level analysis.

In [ ]:
by_symbol = (
    positions.groupby("symbol", as_index=False)
    .agg(
        portfolio_weight=("portfolio_weight", "sum"),
        model_count=("model", "nunique"),
        models=("model", lambda values: ", ".join(sorted(set(values.dropna())))),
    )
    .sort_values("portfolio_weight", ascending=False)
    .reset_index(drop=True)
)
by_symbol["weight_pct"] = by_symbol["portfolio_weight"] * 100
by_symbol.head(10)

In [ ]:
multi_model_symbols = by_symbol.loc[
    by_symbol["model_count"] > 1,
    ["symbol", "model_count", "models", "weight_pct"],
]
multi_model_symbols

## Concentration metrics

In [ ]:
hhi = by_symbol["portfolio_weight"].pow(2).sum()
metrics = pd.Series({
    "unique_positions": len(by_symbol),
    "top_5_weight_pct": by_symbol.head(5)["weight_pct"].sum(),
    "top_10_weight_pct": by_symbol.head(10)["weight_pct"].sum(),
    "hhi": hhi,
    "effective_number_of_positions": 1 / hhi,
})
metrics.round(4)

## Visual analysis

In [ ]:
top_10 = by_symbol.head(10).sort_values("weight_pct")
ax = top_10.plot.barh(x="symbol", y="weight_pct", legend=False, figsize=(9, 5), color="#2563eb")
ax.set(title="Top 10 portfolio positions", xlabel="Portfolio weight (%)", ylabel="")
plt.tight_layout()
plt.show()

In [ ]:
concentration = by_symbol["weight_pct"].cumsum()
ax = concentration.plot(figsize=(9, 5), color="#0f766e")
ax.set(title="Cumulative portfolio concentration", xlabel="Number of positions", ylabel="Cumulative weight (%)")
ax.axhline(80, color="#dc2626", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

## Interpretation checklist

- Compare unique symbols with the effective number of positions.
- Evaluate how much capital the Top 5 and Top 10 control.
- Identify whether concentration comes from securities, models, or both.
- Treat zero-weight residual positions separately before drawing conclusions.